In [ ]:
from math import ceil
from enum import Enum
from dataclasses import dataclass, field
from typing import Optional, List, Dict, Any

class AttentionType(Enum):
    MHA = "multi_head_attention"
    MQA = "multi_query_attention"
    GQA = "grouped_query_attention"
    MLA = "multi_head_latent_attention"
    SWA = "sliding_window_attention"
    HYBRID = "hybrid_attention"

@dataclass
class MLAConfig:
    """Configuration for Multi-head Latent Attention"""
    latent_dimension: int = 1536
    rope_decoupled: bool = True
    matrix_absorption: bool = True

@dataclass
class SWAConfig:
    """Configuration for Sliding Window Attention"""
    window_size: int = 4096
    rotating_buffer: bool = True
    hybrid_layers: bool = False  # For Gemma-style alternating

@dataclass
class HybridConfig:
    """Configuration for Hybrid Attention patterns"""
    swa_layers: List[int] = field(default_factory=list)
    mla_layers: List[int] = field(default_factory=list)
    full_layers: List[int] = field(default_factory=list)
    window_size: int = 4096
    latent_dimension: Optional[int] = None

@dataclass
class ArchitectureConfig:
    """Model architecture configuration"""
    layers: int = 32
    hidden_size: int = 4096
    attention_heads: int = 32
    kv_heads: Optional[int] = None  # None means same as attention_heads
    head_dim: Optional[int] = None  # None means hidden_size / attention_heads

@dataclass
class VLLMOptimizationConfig:
    """vLLM-specific optimization settings"""
    block_size: int = 16
    memory_pool_overhead: float = 0.15
    paged_attention: bool = True
    attention_backend: str = "FLASH_ATTN"

@dataclass
class VRAMRequirementConfig:
    """VRAM calculation parameters"""
    activation_multiplier: float = 1.5
    overhead_factor: float = 1.2
    system_overhead_rate: float = 0.1

@dataclass
class ModelConfig:
    """Complete model configuration for VRAM calculation"""
    # Basic parameters
    name: str = "generic_model"
    parameters: int = 7_000_000_000  # 7B default
    
    # Attention configuration
    attention_type: AttentionType = AttentionType.MHA
    
    # Architecture
    architecture: ArchitectureConfig = field(default_factory=ArchitectureConfig)
    
    # Attention-specific configs
    mla: Optional[MLAConfig] = None
    swa: Optional[SWAConfig] = None
    hybrid: Optional[HybridConfig] = None
    
    # vLLM optimizations
    vllm_optimizations: VLLMOptimizationConfig = field(default_factory=VLLMOptimizationConfig)
    
    # VRAM requirements
    vram_requirements: VRAMRequirementConfig = field(default_factory=VRAMRequirementConfig)
    
    @property
    def overhead_factor(self):
        return self.vram_requirements.overhead_factor
    
    @property
    def attention(self):
        """Convenience property for attention access"""
        return type('Attention', (), {'type': self.attention_type})()

def calculate_vllm_vram(model: ModelConfig, sequence_length: int, batch_size: int, precision_bytes: int):
    """
    Enhanced VRAM calculation supporting multiple attention mechanisms.
    """
    
    # 1. Base Model Memory (unchanged)
    base_memory = model.parameters * precision_bytes * model.overhead_factor
    
    # 2. KV-Cache Memory (attention-specific)
    attention_type = model.attention.type
    
    if attention_type == AttentionType.MHA:
        kv_cache_total = calculate_mha_kv_cache(model, sequence_length, batch_size, precision_bytes)
    elif attention_type == AttentionType.GQA:
        kv_cache_total = calculate_gqa_kv_cache(model, sequence_length, batch_size, precision_bytes)
    elif attention_type == AttentionType.MLA:
        kv_cache_total = calculate_mla_kv_cache(model, sequence_length, batch_size, precision_bytes)
    elif attention_type == AttentionType.SWA:
        kv_cache_total = calculate_swa_kv_cache(model, sequence_length, batch_size, precision_bytes)
    elif attention_type == AttentionType.HYBRID:
        kv_cache_total = calculate_hybrid_kv_cache(model, sequence_length, batch_size, precision_bytes)
    else:
        raise ValueError(f"Unknown attention type: {attention_type}")
    
    # 3. Activation Memory (with attention-specific factor)
    attention_factors = {
        AttentionType.MHA: 1.0,
        AttentionType.GQA: 1.0,
        AttentionType.MLA: 1.25,  # Matrix absorption overhead
        AttentionType.SWA: 0.85,  # Reduced computation
        AttentionType.HYBRID: 0.95
    }
    
    activation_memory = (model.architecture.hidden_size *
                        sequence_length *
                        batch_size *
                        precision_bytes *
                        model.vram_requirements.activation_multiplier *
                        attention_factors[attention_type])
    
    # 4. System Overhead
    system_overhead = (base_memory + kv_cache_total) * 0.1
    
    # Total VRAM
    total_vram = base_memory + kv_cache_total + activation_memory + system_overhead
    
    return {
        'total_vram_gb': total_vram / (1024**3),
        'base_memory_gb': base_memory / (1024**3),
        'kv_cache_gb': kv_cache_total / (1024**3),
        'activation_gb': activation_memory / (1024**3),
        'overhead_gb': system_overhead / (1024**3),
        'attention_type': attention_type.value
    }

def calculate_mha_kv_cache(model: ModelConfig, sequence_length: int, batch_size: int, precision_bytes: int):
    """Standard Multi-Head Attention KV cache calculation"""
    layers = model.architecture.layers
    hidden_size = model.architecture.hidden_size
    
    # Block allocation for vLLM
    block_size = model.vllm_optimizations.block_size
    blocks_needed = ceil(sequence_length / block_size)
    effective_seq_length = blocks_needed * block_size
    
    kv_cache_base = (2 * hidden_size * layers * effective_seq_length * 
                     batch_size * precision_bytes)
    
    memory_pool_overhead = kv_cache_base * model.vllm_optimizations.memory_pool_overhead
    return kv_cache_base + memory_pool_overhead

def calculate_gqa_kv_cache(model: ModelConfig, sequence_length: int, batch_size: int, precision_bytes: int):
    """Grouped Query Attention KV cache calculation"""
    layers = model.architecture.layers
    kv_heads = model.architecture.kv_heads or model.architecture.attention_heads
    head_dim = model.architecture.head_dim or (model.architecture.hidden_size / model.architecture.attention_heads)
    
    # Block allocation for vLLM
    block_size = model.vllm_optimizations.block_size
    blocks_needed = ceil(sequence_length / block_size)
    effective_seq_length = blocks_needed * block_size
    
    kv_cache_base = (2 * layers * kv_heads * head_dim * effective_seq_length * 
                     batch_size * precision_bytes)
    
    memory_pool_overhead = kv_cache_base * model.vllm_optimizations.memory_pool_overhead
    return kv_cache_base + memory_pool_overhead

def calculate_mla_kv_cache(model: ModelConfig, sequence_length: int, batch_size: int, precision_bytes: int):
    """Multi-head Latent Attention KV cache calculation"""
    if not model.mla:
        raise ValueError("MLA configuration required for MLA attention type")
    
    latent_dim = model.mla.latent_dimension
    layers = model.architecture.layers
    
    # Block allocation for vLLM
    block_size = model.vllm_optimizations.block_size
    blocks_needed = ceil(sequence_length / block_size)
    effective_seq_length = blocks_needed * block_size
    
    # MLA cache stores only latent vectors
    kv_cache_base = (latent_dim * layers * effective_seq_length * 
                     batch_size * precision_bytes)
    
    # Additional overhead for RoPE decoupling
    rope_overhead = kv_cache_base * 0.05 if model.mla.rope_decoupled else 0
    
    # Memory pool overhead (vLLM)
    memory_pool_overhead = (kv_cache_base + rope_overhead) * model.vllm_optimizations.memory_pool_overhead
    
    return kv_cache_base + rope_overhead + memory_pool_overhead

def calculate_swa_kv_cache(model: ModelConfig, sequence_length: int, batch_size: int, precision_bytes: int):
    """Sliding Window Attention KV cache calculation"""
    if not model.swa:
        raise ValueError("SWA configuration required for SWA attention type")
    
    window_size = model.swa.window_size
    layers = model.architecture.layers
    hidden_size = model.architecture.hidden_size
    
    # Effective cache size is limited by window
    effective_cache_size = min(window_size, sequence_length)
    
    # Some models (Gemma 2) use hybrid: SWA on odd layers, full on even
    if model.swa.hybrid_layers:
        swa_layers = layers // 2  # Odd layers
        full_layers = layers - swa_layers  # Even layers
        
        swa_cache = (2 * hidden_size * effective_cache_size * 
                     swa_layers * batch_size * precision_bytes)
        
        full_cache = (2 * hidden_size * sequence_length * 
                      full_layers * batch_size * precision_bytes)
        
        kv_cache_base = swa_cache + full_cache
    else:
        # Pure SWA (Mistral approach)
        kv_cache_base = (2 * hidden_size * effective_cache_size * 
                         layers * batch_size * precision_bytes)
    
    # Rotating buffer overhead
    buffer_overhead = kv_cache_base * 0.1 if model.swa.rotating_buffer else 0
    
    # Memory pool overhead (vLLM)
    memory_pool_overhead = (kv_cache_base + buffer_overhead) * model.vllm_optimizations.memory_pool_overhead
    
    return kv_cache_base + buffer_overhead + memory_pool_overhead

def calculate_hybrid_kv_cache(model: ModelConfig, sequence_length: int, batch_size: int, precision_bytes: int):
    """
    Calculate KV cache for models with mixed attention patterns.
    Example: Gemma 2 uses SWA on odd layers, full attention on even layers.
    """
    if not model.hybrid:
        raise ValueError("Hybrid configuration required for HYBRID attention type")
    
    total_cache = 0
    hidden_size = model.architecture.hidden_size
    
    for layer_idx in range(model.architecture.layers):
        if layer_idx in model.hybrid.swa_layers:
            # SWA layer
            window_size = model.hybrid.window_size
            cache_size = min(window_size, sequence_length)
            layer_cache = (2 * hidden_size * cache_size * 
                          batch_size * precision_bytes)
        elif layer_idx in model.hybrid.mla_layers:
            # MLA layer
            if not model.hybrid.latent_dimension:
                raise ValueError("Latent dimension required for MLA layers in hybrid config")
            layer_cache = (model.hybrid.latent_dimension * sequence_length * 
                          batch_size * precision_bytes)
        else:
            # Full attention layer
            layer_cache = (2 * hidden_size * sequence_length * 
                          batch_size * precision_bytes)
        
        total_cache += layer_cache
    
    # Add vLLM overheads
    memory_pool_overhead = total_cache * model.vllm_optimizations.memory_pool_overhead
    return total_cache + memory_pool_overhead

In [ ]:
# DeepSeek V3 configuration
model_deepseek_v3 = ModelConfig(
    name="DeepSeek-V3",
    parameters=671_000_000_000,
    attention_type=AttentionType.MLA,
    architecture=ArchitectureConfig(
        layers=61,
        hidden_size=7168,
        attention_heads=128
    ),
    mla=MLAConfig(
        latent_dimension=1536,
        rope_decoupled=True,
        matrix_absorption=True
    ),
    vllm_optimizations=VLLMOptimizationConfig(
        attention_backend="FLASHINFER"
    )
)

# Calculate VRAM
context_length = 8192
batch_size = 1
precision_bytes = 2
result = calculate_vllm_vram(model_deepseek_v3, context_length, batch_size, precision_bytes)
print(f"DeepSeek V3 Results (context_length: {context_length}, batch_size: {batch_size}, precision_bytes: {precision_bytes}):")
print(f"  Total VRAM: {result['total_vram_gb']:.2f} GB")
print(f"  Base Memory: {result['base_memory_gb']:.2f} GB")
print(f"  KV Cache: {result['kv_cache_gb']:.2f} GB")
print(f"  Attention Type: {result['attention_type']}")

# Compare to standard MHA:
# MHA KV Cache would be: ~14 GB
# MLA achieves 9.3x reduction!

In [ ]:
# Mistral configuration
model_mistral = ModelConfig(
    name="Mistral-7B",
    parameters=7_240_000_000,
    attention_type=AttentionType.SWA,
    architecture=ArchitectureConfig(
        layers=32,
        hidden_size=4096,
        attention_heads=32,
        kv_heads=8  # Also uses GQA
    ),
    swa=SWAConfig(
        window_size=4096,
        rotating_buffer=True,
        hybrid_layers=False
    )
)

# Calculate VRAM
context_length = 16384
batch_size = 10
precision_bytes = 2
result = calculate_vllm_vram(model_mistral, context_length, batch_size, precision_bytes)
print(f"DeepSeek V3 Results (context_length: {context_length}, batch_size: {batch_size}, precision_bytes: {precision_bytes}):")
print(f"  Total VRAM: {result['total_vram_gb']:.2f} GB")
print(f"  Base Memory: {result['base_memory_gb']:.2f} GB")
print(f"  KV Cache: {result['kv_cache_gb']:.2f} GB (capped at window size)")

# Without SWA (full 16K cache):
# KV Cache would be: 2.56 GB
# SWA achieves 75% reduction for long contexts!

In [ ]:
# Gemma 2 configuration (hybrid)
model_gemma2 = ModelConfig(
    name="Gemma-2-27B",
    parameters=27_000_000_000,
    attention_type=AttentionType.HYBRID,
    architecture=ArchitectureConfig(
        layers=46,
        hidden_size=4608,
        attention_heads=32,
        kv_heads=16  # Also uses GQA
    ),
    hybrid=HybridConfig(
        swa_layers=list(range(1, 46, 2)),  # Odd layers
        full_layers=list(range(0, 46, 2)),  # Even layers
        mla_layers=[],  # No MLA layers
        window_size=4096
    )
)

# Calculate VRAM
context_length = 8192
batch_size = 1
precision_bytes = 2
result = calculate_vllm_vram(model_gemma2, context_length, batch_size, precision_bytes)
print(f"DeepSeek V3 Results (context_length: {context_length}, batch_size: {batch_size}, precision_bytes: {precision_bytes}):")
print(f"  Total VRAM: {result['total_vram_gb']:.2f} GB")
print(f"  Base Memory: {result['base_memory_gb']:.2f} GB")
print(f"  KV Cache (hybrid): {result['kv_cache_gb']:.2f} GB")
print(f"    - SWA layers (23): ~0.43 GB")
print(f"    - Full layers (23): ~0.86 GB")